# Pipeline Gardenia | Etapa 7.5: Consultas em Spark SQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anlbp/Projeto-Gardenia-unifeob/blob/main/recursos/dados/notebook_dados_nao_filtrados/07_5_spark_sql.ipynb)

Complemento do notebook 07. Ele usa a mesma carga e a mesma tabela de fatos, mas faz as análises com **SQL** em vez de DataFrames. No Spark, as duas formas usam o mesmo motor (o otimizador Catalyst), então o resultado é o mesmo; o SQL é só outra forma de escrever.

**O que o notebook faz**

1. Lê os `trusted_*_N.csv` de todos os lotes (como no notebook 07).
2. Registra a tabela de fatos como a **view temporária** `fato_vendas`.
3. Roda 4 consultas que usam recursos de SQL analítico:

| Consulta | Recurso de SQL |
|---|---|
| Receita por trimestre e categoria, com a participação de cada categoria | função de janela com `SUM(...) OVER (PARTITION BY ...)` |
| Receita mensal e variação em relação ao mês anterior | CTE (`WITH`) e `LAG` |
| Os 3 produtos de maior receita em cada categoria | `ROW_NUMBER() OVER (PARTITION BY ...)` e subconsulta |
| Cancelamento por faixa etária | `CASE WHEN` e `HAVING` |

4. Mostra o plano de execução de uma consulta (`EXPLAIN`) e confere duas consultas contra a versão em DataFrame.
5. Grava os resultados em CSV (`sql_*.csv`).

**Lê:** `trusted_clientes_N.csv`, `trusted_produtos_N.csv`, `trusted_compras_N.csv` (saída do notebook 03).
**Gera:** `sql_*.csv`, por padrão em `{DRIVE_PASTA}/spark`.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuração

In [5]:
# ============================ CONFIGURAÇÃO ============================
FONTE = "drive"             # "drive" | "github" | "upload" | "local"
LOTES = None                # None = todos os lotes encontrados. Ou uma lista, ex.: [1, 2]

DRIVE_PASTA = "/content/drive/MyDrive/gardenia"   # o 03 grava os trusted em {DRIVE_PASTA}/saida
PASTA_TRUSTED = f"{DRIVE_PASTA}/saida"

# --- usado quando FONTE == "github" (os trusted versionados no repositório) ---
GITHUB_USUARIO = "Anlbp"
GITHUB_REPO    = "Projeto-Gardenia-unifeob"
GITHUB_BRANCH  = "main"
PASTA_GITHUB   = "recursos/dados/trusted"
LOTES_GITHUB   = [1]        # quais lotes baixar do GitHub

# --- usado quando FONTE == "local" ---
PASTA_LOCAL = "dados_trusted"

# --- saída ---
SALVAR_NO_DRIVE = True      # True: grava em {DRIVE_PASTA}/spark. False: grava em ./spark_saida
# =======================================================================

## 2. Instalação do Spark e leitura dos dados

In [6]:
# O Spark precisa de Java 17 ou mais novo. No Colab, instala (leva cerca de 1 minuto na primeira vez).
import os, shutil, subprocess, sys

if shutil.which("java") is None or "17" not in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr:
    subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    import pyspark
print("pyspark", pyspark.__version__)

pyspark 4.0.4


In [7]:
import urllib.error
import urllib.request

try:
    if FONTE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        PASTA_DADOS = PASTA_TRUSTED

    elif FONTE == "github":
        PASTA_DADOS = "dados_trusted"
        os.makedirs(PASTA_DADOS, exist_ok=True)
        branch = GITHUB_BRANCH.removeprefix("tree/")
        for n in LOTES_GITHUB:
            for t in ["clientes", "produtos", "compras"]:
                nome = f"trusted_{t}_{n}.csv"
                url = f"https://raw.githubusercontent.com/{GITHUB_USUARIO}/{GITHUB_REPO}/{branch}/{PASTA_GITHUB}/{nome}"
                print("baixando", url)
                urllib.request.urlretrieve(url, os.path.join(PASTA_DADOS, nome))

    elif FONTE == "upload":
        from google.colab import files
        PASTA_DADOS = "dados_trusted"
        os.makedirs(PASTA_DADOS, exist_ok=True)
        print("Selecione os 3 arquivos trusted_*_N.csv de cada lote (gerados pelo notebook 03)")
        for nome, conteudo in files.upload().items():
            with open(os.path.join(PASTA_DADOS, nome), "wb") as f:
                f.write(conteudo)

    elif FONTE == "local":
        PASTA_DADOS = PASTA_LOCAL

    else:
        raise ValueError('FONTE deve ser "drive", "github", "upload" ou "local".')
except urllib.error.HTTPError as e:
    raise FileNotFoundError(f"Não consegui baixar o arquivo ({e.code}). Confira usuário, repositório, pasta e lotes.") from None

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Código de apoio

As funções de carga e a tabela de fatos são as mesmas do notebook 07. As consultas SQL ficam no dicionário `CONSULTAS_SQL`, com uma descrição de cada uma.

In [8]:
import glob
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [9]:
TABELAS = ["clientes", "produtos", "compras"]
COR_SEGMENTO = {"VIP": "#C44E52", "Regular": "#DD8452", "Baixo engajamento": "#4C72B0"}


def criar_spark(master="local[*]", memoria=None, particoes=8):
    """Sessão Spark. master="local[*]" usa todos os núcleos da máquina; num cluster seria "spark://host:7077"."""
    b = (SparkSession.builder.master(master).appName("gardenia-etapa7")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", str(particoes))
         .config("spark.sql.execution.arrow.pyspark.enabled", "false"))
    if memoria:
        b = b.config("spark.driver.memory", memoria)
    spark = b.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def lotes_trusted(pasta):
    """Lotes que podem ser lidos: os 3 trusted_*_N.csv existem e, se a pasta usa marcadores
    (trusted_N.pronto, gravados pela VM 2), o marcador também existe."""
    arquivos = set(os.listdir(pasta)) if os.path.isdir(pasta) else set()
    usa_marcador = any(re.fullmatch(r"trusted_\d+\.pronto", n) for n in arquivos)
    lotes = []
    for n in arquivos:
        m = re.fullmatch(r"trusted_clientes_(\d+)\.csv", n)
        if not m:
            continue
        N = int(m.group(1))
        if not all(f"trusted_{t}_{N}.csv" in arquivos for t in TABELAS):
            continue
        if usa_marcador and f"trusted_{N}.pronto" not in arquivos:
            continue
        lotes.append(N)
    return sorted(lotes)


def ler_tabela(spark, pasta, tabela, lotes):
    """Lê trusted_<tabela>_N.csv de cada lote e empilha, com a coluna 'lote'."""
    partes = []
    for n in lotes:
        caminho = os.path.abspath(os.path.join(pasta, f"trusted_{tabela}_{n}.csv"))
        partes.append(spark.read.csv(caminho, header=True, inferSchema=False).withColumn("lote", F.lit(n)))
    df = partes[0]
    for p in partes[1:]:
        df = df.unionByName(p)
    return df


def carregar(spark, pasta, lotes):
    """Devolve clientes, produtos e compras já tipados. A chave de cada registro é (lote, id),
    porque o simulador reinicia os ids a cada ciclo."""
    cli = (ler_tabela(spark, pasta, "clientes", lotes)
           .select("lote", F.col("id").cast("int").alias("cliente_id"), "nome",
                   F.col("idade").cast("int").alias("idade"),
                   F.to_timestamp("data_cadastro").alias("data_cadastro"),
                   (F.col("cadastro_incompleto") == "True").alias("cadastro_incompleto")))
    cli = cli.withColumn("faixa_etaria", F.when(F.col("idade").isNull(), "sem idade")
                         .when(F.col("idade") < 25, "18-24").when(F.col("idade") < 35, "25-34")
                         .when(F.col("idade") < 45, "35-44").when(F.col("idade") < 55, "45-54")
                         .when(F.col("idade") < 65, "55-64").otherwise("65+"))
    prod = (ler_tabela(spark, pasta, "produtos", lotes)
            .select("lote", F.col("id").cast("int").alias("produto_id"), F.col("nome").alias("produto"),
                    "categoria", F.col("preco").cast("double").alias("preco"),
                    F.col("satisfacao_media").cast("double").alias("satisfacao_media")))
    comp = (ler_tabela(spark, pasta, "compras", lotes)
            .select("lote", F.col("id").cast("int").alias("compra_id"),
                    F.col("cliente_id").cast("int").alias("cliente_id"),
                    F.col("produto_id").cast("int").alias("produto_id"),
                    F.col("quantidade").cast("int").alias("quantidade"),
                    F.col("valor_total").cast("double").alias("valor_total"),
                    F.col("desconto").cast("double").alias("desconto"),
                    F.to_timestamp("data_compra").alias("data_compra"),
                    "status", F.col("satisfacao").cast("int").alias("satisfacao")))
    return cli, prod, comp


def montar_fato(cli, prod, comp):
    """Tabela de fatos: cada compra com os dados do produto e do cliente (join por lote + id)."""
    fato = (comp.join(prod.select("lote", "produto_id", "produto", "categoria", "preco"), ["lote", "produto_id"], "inner")
                .join(cli.select("lote", "cliente_id", "idade", "faixa_etaria"), ["lote", "cliente_id"], "inner")
                .withColumn("valor_liquido", F.col("valor_total") - F.col("desconto"))
                .withColumn("mes", F.date_format("data_compra", "yyyy-MM"))
                .withColumn("valida", F.col("status") != "cancelado"))
    return fato

In [10]:
def pdf(df):
    return df.toPandas()

In [11]:
CONSULTAS_SQL = {
    "sql_receita_trimestre_categoria": (
        "Receita líquida por trimestre e categoria, com a participação de cada categoria no trimestre (função de janela).",
        """
        SELECT trimestre, categoria,
               ROUND(SUM(valor_liquido), 2) AS receita_liquida,
               ROUND(100 * SUM(valor_liquido) / SUM(SUM(valor_liquido)) OVER (PARTITION BY trimestre), 1) AS pct_do_trimestre
        FROM fato_vendas
        WHERE valida
        GROUP BY trimestre, categoria
        ORDER BY trimestre, receita_liquida DESC
        """),
    "sql_variacao_mensal": (
        "Receita mensal e variação percentual em relação ao mês anterior (LAG).",
        """
        WITH mensal AS (
            SELECT mes, SUM(valor_liquido) AS receita
            FROM fato_vendas WHERE valida GROUP BY mes
        )
        SELECT mes, ROUND(receita, 2) AS receita_liquida,
               ROUND(100 * (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes), 1) AS variacao_pct
        FROM mensal ORDER BY mes
        """),
    "sql_top3_produtos_por_categoria": (
        "Os 3 produtos de maior receita em cada categoria (ROW_NUMBER por categoria).",
        """
        SELECT * FROM (
            SELECT categoria, lote, produto_id, produto,
                   ROUND(SUM(valor_liquido), 2) AS receita_liquida,
                   ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(valor_liquido) DESC) AS posicao
            FROM fato_vendas WHERE valida
            GROUP BY categoria, lote, produto_id, produto
        ) WHERE posicao <= 3
        ORDER BY categoria, posicao
        """),
    "sql_cancelamento_faixa_etaria": (
        "Taxa de cancelamento por faixa etária (só faixas com pelo menos 100 compras).",
        """
        SELECT faixa_etaria,
               COUNT(*) AS compras,
               SUM(CASE WHEN valida THEN 0 ELSE 1 END) AS canceladas,
               ROUND(100 * AVG(CASE WHEN valida THEN 0.0 ELSE 1.0 END), 1) AS pct_cancelamento
        FROM fato_vendas
        GROUP BY faixa_etaria
        HAVING COUNT(*) >= 100
        ORDER BY pct_cancelamento DESC
        """),
}


def consultas_sql(fato):
    """Registra a tabela de fatos como view temporária e roda as consultas em Spark SQL."""
    spark = fato.sparkSession
    fato.withColumn("trimestre", F.concat(F.year("data_compra").cast("string"), F.lit("-T"),
                                          F.quarter("data_compra").cast("string"))) \
        .createOrReplaceTempView("fato_vendas")
    return {nome: pdf(spark.sql(sql)) for nome, (_, sql) in CONSULTAS_SQL.items()}

## 4. Carga e tabela de fatos

In [12]:
spark = criar_spark("local[*]")
lotes = lotes_trusted(PASTA_DADOS)
if LOTES is not None:
    lotes = [n for n in lotes if n in LOTES]
if not lotes:
    raise FileNotFoundError(
        f"Não encontrei lotes completos em {PASTA_DADOS}.\n"
        "Rode antes o notebook 03 (ele grava trusted_clientes/produtos/compras_N.csv) e confira FONTE e as pastas na configuração.")
print("Lotes lidos:", lotes)

cli, prod, comp = carregar(spark, PASTA_DADOS, lotes)
fato = montar_fato(cli, prod, comp).cache()
print(f"fato_vendas: {fato.count()} compras")
fato.printSchema()

Lotes lidos: [1]
fato_vendas: 1184 compras
root
 |-- lote: integer (nullable = false)
 |-- cliente_id: integer (nullable = true)
 |-- produto_id: integer (nullable = true)
 |-- compra_id: integer (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- desconto: double (nullable = true)
 |-- data_compra: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- satisfacao: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: double (nullable = true)
 |-- idade: integer (nullable = true)
 |-- faixa_etaria: string (nullable = false)
 |-- valor_liquido: double (nullable = true)
 |-- mes: string (nullable = true)
 |-- valida: boolean (nullable = true)



## 5. As consultas

`consultas_sql` registra a view `fato_vendas` (acrescentando a coluna `trimestre`) e roda as 4 consultas. Para cada uma o notebook mostra a descrição, o SQL e o resultado.

In [13]:
import inspect
sql_res = consultas_sql(fato)
for nome, d in sql_res.items():
    descricao, sql = CONSULTAS_SQL[nome]
    print(f"\n=== {nome}\n{descricao}\n{inspect.cleandoc(sql)}\n")
    display(d.head(15))


=== sql_receita_trimestre_categoria
Receita líquida por trimestre e categoria, com a participação de cada categoria no trimestre (função de janela).
SELECT trimestre, categoria,
       ROUND(SUM(valor_liquido), 2) AS receita_liquida,
       ROUND(100 * SUM(valor_liquido) / SUM(SUM(valor_liquido)) OVER (PARTITION BY trimestre), 1) AS pct_do_trimestre
FROM fato_vendas
WHERE valida
GROUP BY trimestre, categoria
ORDER BY trimestre, receita_liquida DESC



,trimestre,categoria,receita_liquida,pct_do_trimestre
0,2025-T4,Livros,85557.73,18.1
1,2025-T4,Roupas,75679.19,16.0
2,2025-T4,Esportes,72815.16,15.4
3,2025-T4,Alimentos,72265.06,15.3
4,2025-T4,Eletrônicos,71170.04,15.1
5,2025-T4,Casa,58478.03,12.4
6,2025-T4,Brinquedos,36276.75,7.7
7,2026-T1,Esportes,76638.02,17.2
8,2026-T1,Casa,75684.62,17.0
9,2026-T1,Livros,65372.12,14.7



=== sql_variacao_mensal
Receita mensal e variação percentual em relação ao mês anterior (LAG).
WITH mensal AS (
    SELECT mes, SUM(valor_liquido) AS receita
    FROM fato_vendas WHERE valida GROUP BY mes
)
SELECT mes, ROUND(receita, 2) AS receita_liquida,
       ROUND(100 * (receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes), 1) AS variacao_pct
FROM mensal ORDER BY mes



,mes,receita_liquida,variacao_pct
0,2025-10,151704.07,NaN
1,2025-11,168367.12,11.0
2,2025-12,152170.77,-9.6
3,2026-01,150093.82,-1.4
4,2026-02,115549.23,-23.0
5,2026-03,179081.66,55.0
6,2026-04,156521.73,-12.6
7,2026-05,158326.12,1.2
8,2026-06,145023.52,-8.4
9,2026-07,209200.72,44.3



=== sql_top3_produtos_por_categoria
Os 3 produtos de maior receita em cada categoria (ROW_NUMBER por categoria).
SELECT * FROM (
    SELECT categoria, lote, produto_id, produto,
           ROUND(SUM(valor_liquido), 2) AS receita_liquida,
           ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY SUM(valor_liquido) DESC) AS posicao
    FROM fato_vendas WHERE valida
    GROUP BY categoria, lote, produto_id, produto
) WHERE posicao <= 3
ORDER BY categoria, posicao



,categoria,lote,produto_id,produto,receita_liquida,posicao
0,Alimentos,1,752,Camiseta Premium,8396.06,1
1,Alimentos,1,2076,Notebook Plus,8090.77,2
2,Alimentos,1,4462,Camiseta Pro,7033.99,3
3,Brinquedos,1,4468,Smartphone Pro,8851.72,1
4,Brinquedos,1,614,Notebook Max,8521.55,2
5,Brinquedos,1,1695,Ursinho Pro,7226.09,3
6,Casa,1,4252,Aventura Basic,7603.89,1
7,Casa,1,2422,Smartphone Max,7018.01,2
8,Casa,1,1809,Romance Max,6761.80,3
9,Eletrônicos,1,2510,Sofá Pro,8011.57,1



=== sql_cancelamento_faixa_etaria
Taxa de cancelamento por faixa etária (só faixas com pelo menos 100 compras).
SELECT faixa_etaria,
       COUNT(*) AS compras,
       SUM(CASE WHEN valida THEN 0 ELSE 1 END) AS canceladas,
       ROUND(100 * AVG(CASE WHEN valida THEN 0.0 ELSE 1.0 END), 1) AS pct_cancelamento
FROM fato_vendas
GROUP BY faixa_etaria
HAVING COUNT(*) >= 100
ORDER BY pct_cancelamento DESC



,faixa_etaria,compras,canceladas,pct_cancelamento
0,45-54,210,53,25.2
1,65+,291,63,21.6
2,18-24,129,27,20.9
3,25-34,183,38,20.8
4,35-44,169,33,19.5
5,55-64,202,32,15.8


## 6. Interpretação das consultas

Preencha depois de rodar, com os números que apareceram:

- **Trimestre e categoria:** qual categoria lidera em cada trimestre e se a participação muda ou fica parecida (se todas ficam perto de 1/7 = 14%, não há categoria dominante).
- **Variação mensal:** os meses com maior alta e maior queda. O primeiro mês não tem variação (não há mês anterior), e o primeiro e o último mês costumam estar incompletos.
- **Top 3 por categoria:** se os mesmos produtos aparecem com frequência ou se a receita é pulverizada.
- **Cancelamento por faixa etária:** se a diferença entre faixas é grande o bastante para agir (confira com o teste do notebook 08).

## 7. Plano de execução e conferência

In [14]:
# O mesmo motor executa SQL e DataFrames. EXPLAIN mostra o plano que o Spark montou (leia de baixo para cima).
spark.sql("EXPLAIN FORMATTED " + inspect.cleandoc(CONSULTAS_SQL["sql_cancelamento_faixa_etaria"][1])).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
# Conferência: a mesma conta feita com DataFrames precisa dar o mesmo resultado que o SQL.
from pyspark.sql import functions as F

df_canc = (fato.groupBy("faixa_etaria")
           .agg(F.count("*").alias("compras"),
                F.sum(F.when(F.col("valida"), 0).otherwise(1)).alias("canceladas"),
                F.round(100 * F.avg(F.when(F.col("valida"), 0.0).otherwise(1.0)), 1).alias("pct_cancelamento"))
           .filter("compras >= 100").toPandas())
sql_canc = sql_res["sql_cancelamento_faixa_etaria"]
a = df_canc.sort_values("faixa_etaria").reset_index(drop=True)
b = sql_canc.sort_values("faixa_etaria").reset_index(drop=True)[a.columns]
assert a.equals(b) or (a.astype(str).values == b.astype(str).values).all(), "SQL e DataFrame divergem"

mensal_df = (fato.filter("valida").groupBy("mes").agg(F.round(F.sum("valor_liquido"), 2).alias("receita_liquida"))
             .orderBy("mes").toPandas())
mensal_sql = sql_res["sql_variacao_mensal"][["mes", "receita_liquida"]]
assert np.allclose(mensal_df["receita_liquida"], mensal_sql["receita_liquida"]), "receita mensal diverge"
print("Conferência: SQL e DataFrame dão o mesmo resultado (cancelamento por faixa e receita mensal).")

Conferência: SQL e DataFrame dão o mesmo resultado (cancelamento por faixa e receita mensal).


## 8. Salvar resultados

In [16]:
SAIDA = f"{DRIVE_PASTA}/spark" if (SALVAR_NO_DRIVE and FONTE == "drive") else "spark_saida"
os.makedirs(SAIDA, exist_ok=True)
for nome, d in sql_res.items():
    destino = os.path.join(SAIDA, f"{nome}.csv")
    d.to_csv(destino, index=False)
    print(f"  {destino}  ({len(d)} linhas)")
spark.stop()

  /content/drive/MyDrive/gardenia/spark/sql_receita_trimestre_categoria.csv  (35 linhas)
  /content/drive/MyDrive/gardenia/spark/sql_variacao_mensal.csv  (13 linhas)
  /content/drive/MyDrive/gardenia/spark/sql_top3_produtos_por_categoria.csv  (21 linhas)
  /content/drive/MyDrive/gardenia/spark/sql_cancelamento_faixa_etaria.csv  (6 linhas)
